# LMCF — CPU-Only Cross-Experiment Metrics

Reads from saved CSV and JSON files only.

**Outputs (all in `Results/`):**
- `combined_{model}.csv` — all experiments merged per model
- `metrics_summary.csv` — BWT, Retention, FWT, Mixed Gap per model
- `learning_speed.csv` — steps to reach PPL thresholds per stage
- `weight_change.csv` — per-layer weight change norm S1→S2
- `weight_change_summary.csv` — aggregated by layer type
- `metrics_summary.txt` — human-readable report of all above

In [ ]:
import os
import json
import pandas as pd
import numpy as np

LMCF_ROOT = '.'
os.environ["LMCF_ROOT"] = LMCF_ROOT

PROJECT_ROOT = os.environ.get('LMCF_ROOT', os.path.expanduser('~/lmcf_project'))
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'Results')
CKPT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')

MODELS = ['M1', 'M2', 'M3']
EXPERIMENTS = ['E1', 'E2', 'E3']
STAGE_LABELS = {
    'E1': ['after_A', 'after_A_then_B'],
    'E2': ['mixed'],
    'E3': ['after_B', 'after_B_then_A'],
}

print(f'Project  : {PROJECT_ROOT}')
print(f'Results  : {RESULTS_DIR}')

Project  : .
Results  : .\Results


## Load and Combine CSVs per Model

In [ ]:
def load_run(model, exp):
    path = os.path.join(RESULTS_DIR, f'{model}_{exp}_eval.csv')
    if not os.path.exists(path):
        print(f'  Missing: {model}_{exp}_eval.csv')
        return None
    df = pd.read_csv(path)
    latest_ts = df['timestamp'].max()
    df = df[df['timestamp'] == latest_ts].copy()
    print(f'  Loaded {model}_{exp}: {len(df)} rows  (ts={latest_ts})')
    return df

def get_stage_row(df, stage_num):
    if df is None or len(df) == 0:
        return None
    return df.iloc[0] if stage_num == 1 else df.iloc[-1]

def safe(series, col):
    if series is None:
        return np.nan
    val = series.get(col, np.nan)
    return np.nan if pd.isna(val) else float(val)


print('--- Loading CSVs ---')
runs      = {}
available = []
missing   = []

for model in MODELS:
    for exp in EXPERIMENTS:
        df = load_run(model, exp)
        runs[(model, exp)] = df
        (available if df is not None else missing).append(f'{model}_{exp}')

print(f'\nAvailable : {available}')
if missing:
    print(f'Missing : {missing}')
    print('Note: metrics requiring missing runs will show NaN')

--- Loading CSVs ---
  Loaded M1_E1: 2 rows  (ts=2026-04-25T10:43:21)
  Loaded M1_E2: 1 rows  (ts=2026-04-25T09:07:32)
  Loaded M1_E3: 2 rows  (ts=2026-04-25T08:37:25)
  Loaded M2_E1: 2 rows  (ts=2026-04-25T12:09:10)
  Loaded M2_E2: 1 rows  (ts=2026-04-25T12:09:10)
  Loaded M2_E3: 2 rows  (ts=2026-04-25T12:09:10)
  Loaded M3_E1: 2 rows  (ts=2026-04-25T13:41:24)
  Loaded M3_E2: 1 rows  (ts=2026-04-25T14:35:01)
  Loaded M3_E3: 2 rows  (ts=2026-04-25T16:12:14)

Available : ['M1_E1', 'M1_E2', 'M1_E3', 'M2_E1', 'M2_E2', 'M2_E3', 'M3_E1', 'M3_E2', 'M3_E3']


In [3]:
print('--- Combining CSVs per model ---')
for model in MODELS:
    frames = [runs[(model, exp)] for exp in EXPERIMENTS if runs.get((model, exp)) is not None]
    if not frames:
        print(f'  {model}: no data')
        continue
    combined = pd.concat(frames, ignore_index=True)
    out_path = os.path.join(RESULTS_DIR, f'combined_{model}.csv')
    combined.to_csv(out_path, index=False)
    print(f'  {model}: {len(combined)} rows -> {out_path}')

--- Combining CSVs per model ---
  M1: 5 rows -> .\Results\combined_M1.csv
  M2: 5 rows -> .\Results\combined_M2.csv
  M3: 5 rows -> .\Results\combined_M3.csv


## Cross-Experiment Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **BWT** (Backward Transfer / Forgetting) | `E1_s2.ppl_a − E1_s1.ppl_a` | positive = forgetting happened |
| **Retention** | `E1_s1.ppl_a / E1_s2.ppl_a` | 1.0 = perfect, 0 = total forgetting |
| **FWT** (Forward Transfer) | `E1_s2.ppl_b − E3_s1.ppl_b` | negative = pretraining on A helped learn B |
| **Mixed Gap** | `E1_s2.ppl_a − E2_s1.ppl_a` | positive = sequential worse than joint/mixed |

In [4]:
def bwt(after, before):
    return round(after - before, 4) if not (np.isnan(after) or np.isnan(before)) else np.nan

def bwt_norm(after, before):
    """Normalised BWT = (after - before) / before. Scale-independent forgetting ratio."""
    return round((after - before) / before, 4) if not (
        np.isnan(after) or np.isnan(before) or before == 0) else np.nan

def ret(before, after):
    return round(before / after, 4) if not (np.isnan(before) or np.isnan(after) or after == 0) else np.nan

def fwt(with_pretrain, from_scratch):
    return round(with_pretrain - from_scratch, 4) if not (np.isnan(with_pretrain) or np.isnan(from_scratch)) else np.nan

def fwt_norm(with_pretrain, from_scratch):
    """Normalised FWT = (with_pretrain - from_scratch) / from_scratch.
    Negative = pretraining helped (scale-independent). Use for cross-model plots."""
    return round((with_pretrain - from_scratch) / from_scratch, 4) if not (
        np.isnan(with_pretrain) or np.isnan(from_scratch) or from_scratch == 0) else np.nan

def gap(sequential, mixed):
    return round(sequential - mixed, 4) if not (np.isnan(sequential) or np.isnan(mixed)) else np.nan


print('\n--- Computing cross-experiment metrics ---')
metric_rows = []

for model in MODELS:
    e1_s1 = get_stage_row(runs.get((model, 'E1')), 1)
    e1_s2 = get_stage_row(runs.get((model, 'E1')), 2)
    e2_s1 = get_stage_row(runs.get((model, 'E2')), 1)
    e3_s1 = get_stage_row(runs.get((model, 'E3')), 1)
    e3_s2 = get_stage_row(runs.get((model, 'E3')), 2)

    # Val PPL
    e1s1_a = safe(e1_s1, 'ppl_a');  e1s1_b = safe(e1_s1, 'ppl_b')
    e1s2_a = safe(e1_s2, 'ppl_a');  e1s2_b = safe(e1_s2, 'ppl_b')
    e2s1_a = safe(e2_s1, 'ppl_a');  e2s1_b = safe(e2_s1, 'ppl_b')
    e3s1_a = safe(e3_s1, 'ppl_a');  e3s1_b = safe(e3_s1, 'ppl_b')
    e3s2_a = safe(e3_s2, 'ppl_a');  e3s2_b = safe(e3_s2, 'ppl_b')

    # Test PPL
    e1s1_at = safe(e1_s1, 'ppl_a_test'); e1s1_bt = safe(e1_s1, 'ppl_b_test')
    e1s2_at = safe(e1_s2, 'ppl_a_test'); e1s2_bt = safe(e1_s2, 'ppl_b_test')
    e2s1_at = safe(e2_s1, 'ppl_a_test'); e2s1_bt = safe(e2_s1, 'ppl_b_test')
    e3s1_at = safe(e3_s1, 'ppl_a_test'); e3s1_bt = safe(e3_s1, 'ppl_b_test')
    e3s2_at = safe(e3_s2, 'ppl_a_test'); e3s2_bt = safe(e3_s2, 'ppl_b_test')

    row = {
        'model': model,
        # Raw PPL reference
        'E1_s1_ppl_a': e1s1_a,  'E1_s1_ppl_b': e1s1_b,
        'E1_s2_ppl_a': e1s2_a,  'E1_s2_ppl_b': e1s2_b,
        'E2_ppl_a':    e2s1_a,  'E2_ppl_b':    e2s1_b,
        'E3_s1_ppl_a': e3s1_a,  'E3_s1_ppl_b': e3s1_b,
        'E3_s2_ppl_a': e3s2_a,  'E3_s2_ppl_b': e3s2_b,
        # BWT raw
        'BWT_A_val':        bwt(e1s2_a,  e1s1_a),
        'BWT_B_val':        bwt(e3s2_b,  e3s1_b),
        'BWT_A_test':       bwt(e1s2_at, e1s1_at),
        'BWT_B_test':       bwt(e3s2_bt, e3s1_bt),
        # BWT normalised = (PPL_S2 - PPL_S1) / PPL_S1  (scale-independent, use for cross-model plots)
        'BWT_A_norm_val':   bwt_norm(e1s2_a,  e1s1_a),
        'BWT_B_norm_val':   bwt_norm(e3s2_b,  e3s1_b),
        'BWT_A_norm_test':  bwt_norm(e1s2_at, e1s1_at),
        'BWT_B_norm_test':  bwt_norm(e3s2_bt, e3s1_bt),
        # Retention
        'Ret_A_val':   ret(e1s1_a,  e1s2_a),
        'Ret_B_val':   ret(e3s1_b,  e3s2_b),
        'Ret_A_test':  ret(e1s1_at, e1s2_at),
        'Ret_B_test':  ret(e3s1_bt, e3s2_bt),
        # FWT raw
        'FWT_B_val':        fwt(e1s2_b,  e3s1_b),
        'FWT_A_val':        fwt(e3s2_a,  e1s1_a),
        'FWT_B_test':       fwt(e1s2_bt, e3s1_bt),
        'FWT_A_test':       fwt(e3s2_at, e1s1_at),
        # FWT normalised = (with_pretrain - from_scratch) / from_scratch
        'FWT_B_norm_val':   fwt_norm(e1s2_b,  e3s1_b),
        'FWT_A_norm_val':   fwt_norm(e3s2_a,  e1s1_a),
        'FWT_B_norm_test':  fwt_norm(e1s2_bt, e3s1_bt),
        'FWT_A_norm_test':  fwt_norm(e3s2_at, e1s1_at),
        # Mixed Gap
        'Gap_A_val':   gap(e1s2_a,  e2s1_a),
        'Gap_B_val':   gap(e3s2_b,  e2s1_b),
        'Gap_A_test':  gap(e1s2_at, e2s1_at),
        'Gap_B_test':  gap(e3s2_bt, e2s1_bt),
    }
    metric_rows.append(row)

    print(f'\n  {model}:')
    print(f'    BWT_A_val      = {row["BWT_A_val"]:+.2f}   (positive = forgetting)')
    print(f'    BWT_A_norm_val = {row["BWT_A_norm_val"]:+.4f}  (ratio, use for cross-model plots)')
    print(f'    BWT_B_val      = {row["BWT_B_val"]:+.2f}')
    print(f'    BWT_B_norm_val = {row["BWT_B_norm_val"]:+.4f}')
    print(f'    Ret_A_val = {row["Ret_A_val"]:.4f}  (1.0 = perfect retention)')
    print(f'    Ret_B_val = {row["Ret_B_val"]:.4f}')
    print(f'    FWT_B_val      = {row["FWT_B_val"]:+.2f}   (negative = positive transfer)')
    print(f'    FWT_B_norm_val = {row["FWT_B_norm_val"]:+.4f}  (ratio, use for cross-model plots)')
    print(f'    FWT_A_val      = {row["FWT_A_val"]:+.2f}')
    print(f'    FWT_A_norm_val = {row["FWT_A_norm_val"]:+.4f}')
    print(f'    Gap_A_val = {row["Gap_A_val"]:+.2f}   (sequential vs mixed)')
    print(f'    Gap_B_val = {row["Gap_B_val"]:+.2f}')



--- Computing cross-experiment metrics ---

  M1:
    BWT_A_val      = +393.85   (positive = forgetting)
    BWT_A_norm_val = +19.9513  (ratio, use for cross-model plots)
    BWT_B_val      = +5783.04
    BWT_B_norm_val = +50.0524
    Ret_A_val = 0.0477  (1.0 = perfect retention)
    Ret_B_val = 0.0196
    FWT_B_val      = -22.84   (negative = positive transfer)
    FWT_B_norm_val = -0.1977  (ratio, use for cross-model plots)
    FWT_A_val      = -3.67
    FWT_A_norm_val = -0.1857
    Gap_A_val = +388.24   (sequential vs mixed)
    Gap_B_val = +5748.27

  M2:
    BWT_A_val      = +246.75   (positive = forgetting)
    BWT_A_norm_val = +19.3870  (ratio, use for cross-model plots)
    BWT_B_val      = +6941.76
    BWT_B_norm_val = +95.0113
    Ret_A_val = 0.0491  (1.0 = perfect retention)
    Ret_B_val = 0.0104
    FWT_B_val      = -19.66   (negative = positive transfer)
    FWT_B_norm_val = -0.2690  (ratio, use for cross-model plots)
    FWT_A_val      = -2.44
    FWT_A_norm_val = -0.19

## Save Metrics Summary

In [6]:
metrics_df = pd.DataFrame(metric_rows)

csv_path = os.path.join(RESULTS_DIR, 'metrics_summary.csv')
metrics_df.to_csv(csv_path, index=False)
print(f'\nSaved -> {csv_path}')

txt_path = os.path.join(RESULTS_DIR, 'metrics_summary.txt')
with open(txt_path, 'w') as f:
    f.write('=' * 70 + '\n')
    f.write('LMCF - Cross-Experiment Metrics Summary\n')
    f.write('=' * 70 + '\n\n')

    sections = [
        ('BWT raw - Backward Transfer / Forgetting  (positive = worse)',
         ['BWT_A_val', 'BWT_A_test', 'BWT_B_val', 'BWT_B_test']),
        ('BWT normalised = (PPL_S2 - PPL_S1) / PPL_S1  (use for cross-model comparison)',
         ['BWT_A_norm_val', 'BWT_A_norm_test', 'BWT_B_norm_val', 'BWT_B_norm_test']),
        ('Retention  (1.0 = perfect,  lower = more forgetting)',
         ['Ret_A_val', 'Ret_A_test', 'Ret_B_val', 'Ret_B_test']),
        ('FWT raw - Forward Transfer  (negative = pretraining helped)',
         ['FWT_B_val', 'FWT_B_test', 'FWT_A_val', 'FWT_A_test']),
        ('FWT normalised = (with_pretrain - from_scratch) / from_scratch',
         ['FWT_B_norm_val', 'FWT_B_norm_test', 'FWT_A_norm_val', 'FWT_A_norm_test']),
        ('Mixed Gap - Sequential vs Simultaneous  (positive = sequential worse)',
         ['Gap_A_val', 'Gap_A_test', 'Gap_B_val', 'Gap_B_test']),
        ('Raw PPL Reference',
         ['E1_s1_ppl_a', 'E1_s2_ppl_a', 'E1_s2_ppl_b',
          'E2_ppl_a',    'E2_ppl_b',
          'E3_s1_ppl_b', 'E3_s2_ppl_a', 'E3_s2_ppl_b']),
    ]

    for title, cols in sections:
        f.write(f'\n{title}\n')
        f.write('-' * 60 + '\n')
        available_cols = [c for c in cols if c in metrics_df.columns]
        f.write(metrics_df[['model'] + available_cols].to_string(index=False))
        f.write('\n')

print(f'Saved -> {txt_path}')

print('\n' + '=' * 70)
print('METRICS SUMMARY')
print('=' * 70)
key_cols      = ['model', 'BWT_A_val', 'BWT_B_val',
                 'Ret_A_val', 'Ret_B_val',
                 'FWT_B_val', 'FWT_A_val',
                 'Gap_A_val', 'Gap_B_val']
available_key = [c for c in key_cols if c in metrics_df.columns]
print(metrics_df[available_key].to_string(index=False))



Saved -> .\Results\metrics_summary.csv
Saved -> .\Results\metrics_summary.txt

METRICS SUMMARY
model  BWT_A_val  BWT_B_val  Ret_A_val  Ret_B_val  FWT_B_val  FWT_A_val  Gap_A_val  Gap_B_val
   M1   393.8530  5783.0373     0.0477     0.0196   -22.8435    -3.6655   388.2352  5748.2744
   M2   246.7543  6941.7593     0.0491     0.0104   -19.6562    -2.4432   242.4135  6914.1554
   M3   182.9669  2808.3688     0.0624     0.0264   -25.6248    -2.6162   178.2794  2778.9696


## Learning Speed

Reads `training_history` from JSON logs.  
Reports the first step where val PPL dropped below each threshold.  
Answers **RQ3**: do larger models converge faster?

In [19]:
print('=' * 70)
print('LEARNING SPEED - Steps to reach PPL threshold')
print('=' * 70)

PPL_TARGETS = [500, 200, 100, 50, 20]

def steps_to_ppl(steps, val_ppls, target):
    for step, ppl in zip(steps, val_ppls):
        if ppl <= target:
            return step
    return None

speed_rows = []

for model in MODELS:
    for exp in EXPERIMENTS:
        json_path = os.path.join(RESULTS_DIR, f'{model}_{exp}_runs.json')
        if not os.path.exists(json_path):
            continue
        with open(json_path) as fh:
            run_log = json.load(fh)
        latest = run_log[sorted(run_log.keys())[-1]]

        for lbl in STAGE_LABELS[exp]:
            sd       = latest.get(lbl, {})
            hist     = sd.get('training_history', {})
            steps    = hist.get('steps',   [])
            val_ppls = hist.get('val_ppl', [])
            if not steps or not val_ppls:
                continue
            row = {
                'model':         model,
                'experiment':    exp,
                'stage':         lbl,
                'final_val_ppl': round(val_ppls[-1], 2),
                'steps_trained': steps[-1],
            }
            for target in PPL_TARGETS:
                row[f'steps_to_ppl{target}'] = steps_to_ppl(steps, val_ppls, target)
            speed_rows.append(row)

if speed_rows:
    speed_df   = pd.DataFrame(speed_rows)
    speed_path = os.path.join(RESULTS_DIR, 'learning_speed.csv')
    speed_df.to_csv(speed_path, index=False)
    print(speed_df.to_string(index=False))
    print(f'\nSaved -> {speed_path}')
    with open(txt_path, 'a') as f:
        f.write('\n\n' + '=' * 70 + '\n')
        f.write('Learning Speed - Steps to Reach PPL Threshold\n')
        f.write('-' * 60 + '\n')
        f.write(speed_df.to_string(index=False))
        f.write('\n')
else:
    print('No JSON training history found - run training first')

LEARNING SPEED - Steps to reach PPL threshold
model experiment          stage  final_val_ppl  steps_trained  steps_to_ppl500  steps_to_ppl200  steps_to_ppl100  steps_to_ppl50  steps_to_ppl20
   M1         E1        after_A          19.74           5000              400              600            800.0          1000.0          4200.0
   M1         E1 after_A_then_B          92.70           5000              400             1000           3000.0             NaN             NaN
   M1         E2          mixed          66.19           5000              600             1000           1800.0             NaN             NaN
   M1         E3        after_B         115.54           5000              800             1800              NaN             NaN             NaN
   M1         E3 after_B_then_A          16.08           5000              200              200            200.0           400.0          1600.0
   M2         E1        after_A          16.33           5000              400      

## Weight Change Magnitude

Loads Stage 1 and Stage 2 best checkpoints.  
Per-layer weight change norm shows **where** in the model forgetting occurs.

In [18]:
print('=' * 70)
print('WEIGHT CHANGE MAGNITUDE - Per layer  (Stage 1 -> Stage 2 best checkpoints)')
print('=' * 70)

try:
    import torch

    ckpt_pairs = {
        'E1': ('after_A',  'after_A_then_B'),
        'E3': ('after_B',  'after_B_then_A'),
    }
    weight_rows = []

    for model in MODELS:
        for exp, (lbl1, lbl2) in ckpt_pairs.items():
            p1 = os.path.join(CKPT_DIR, f'{model}_{exp}_best_{lbl1}.pt')
            p2 = os.path.join(CKPT_DIR, f'{model}_{exp}_best_{lbl2}.pt')
            if not os.path.exists(p1) or not os.path.exists(p2):
                print(f'  {model}_{exp}: checkpoints not found - skipping')
                continue

            s1 = torch.load(p1, map_location='cpu')['model_state']
            s2 = torch.load(p2, map_location='cpu')['model_state']

            print(f'\n  {model}_{exp}: {lbl1} -> {lbl2}')
            print(f'  {"Layer":<40} {"Norm delta":>10} {"Rel delta":>10}')
            print(f'  {"-"*40} {"-"*10} {"-"*10}')

            for name in s1:
                if name not in s2:
                    continue
                p1t  = s1[name].float()
                p2t  = s2[name].float()
                diff = (p2t - p1t).norm().item()
                base = p1t.norm().item()
                rel  = round(diff / base, 6) if base > 0 else 0.0

                if   'token_emb' in name or 'pos_emb' in name: ltype = 'embedding'
                elif 'lm_head'   in name:                       ltype = 'lm_head'
                elif 'attn'      in name or 'qkv' in name \
                  or 'out_proj'  in name:                       ltype = 'attention'
                elif 'ff'        in name or 'ffn' in name:     ltype = 'ffn'
                elif 'norm'      in name:                       ltype = 'layernorm'
                else:                                           ltype = 'other'

                weight_rows.append({
                    'model':       model,
                    'experiment':  exp,
                    'layer':       name,
                    'layer_type':  ltype,
                    'norm_change': round(diff, 6),
                    'rel_change':  rel,
                })
                print(f'  {name:<40} {diff:>10.4f} {rel:>10.4f}')

            local_rows = [r for r in weight_rows
                          if r['model'] == model and r['experiment'] == exp]
            if local_rows:
                df_l = pd.DataFrame(local_rows)
                agg  = df_l.groupby('layer_type')['rel_change'] \
                           .mean().sort_values(ascending=False)
                print(f'\n  Avg relative change by layer type:')
                for ltype, val in agg.items():
                    print(f'    {ltype:<15} {val:.4f}')

    if weight_rows:
        wc_df    = pd.DataFrame(weight_rows)
        wc_path  = os.path.join(RESULTS_DIR, 'weight_change.csv')
        wc_df.to_csv(wc_path, index=False)
        print(f'\nSaved -> {wc_path}')

        agg_df   = wc_df.groupby(['model', 'experiment', 'layer_type'])['rel_change'] \
                        .mean().reset_index()
        agg_df.columns = ['model', 'experiment', 'layer_type', 'avg_rel_change']
        agg_path = os.path.join(RESULTS_DIR, 'weight_change_summary.csv')
        agg_df.to_csv(agg_path, index=False)
        print(f'Saved -> {agg_path}')

        with open(txt_path, 'a') as f:
            f.write('\n\n' + '=' * 70 + '\n')
            f.write('Weight Change by Layer Type (avg relative change S1 -> S2)\n')
            f.write('-' * 60 + '\n')
            f.write(agg_df.to_string(index=False))
            f.write('\n')
    else:
        print('No checkpoint pairs found')

except ImportError:
    print('PyTorch not installed - weight change analysis skipped')

print('\nDone.')

WEIGHT CHANGE MAGNITUDE - Per layer  (Stage 1 -> Stage 2 best checkpoints)


C:\Users\rajcr\AppData\Local\Temp\ipykernel_31388\2332656341.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s1 = torch.load(p1, map_location='cpu')['model_state']
C:\U


  M1_E1: after_A -> after_A_then_B
  Layer                                    Norm delta  Rel delta
  ---------------------------------------- ---------- ----------
  token_emb.weight                            79.6399     0.5115
  pos_emb.weight                               5.0704     0.8664
  blocks.0.norm1.weight                        1.1381     0.1211
  blocks.0.norm1.bias                          0.5694     0.8473
  blocks.0.attn.qkv.weight                     6.2266     0.5082
  blocks.0.attn.out_proj.weight                2.4166     0.2443
  blocks.0.norm2.weight                        0.6798     0.0622
  blocks.0.norm2.bias                          0.1576     0.5718
  blocks.0.ff.0.weight                         7.6282     0.4982
  blocks.0.ff.0.bias                           0.4324     0.3591
  blocks.0.ff.3.weight                         7.9819     0.6331
  blocks.0.ff.3.bias                           0.1316     0.4912
  blocks.1.norm1.weight                        0.8024 

C:\Users\rajcr\AppData\Local\Temp\ipykernel_31388\2332656341.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s1 = torch.load(p1, map_location='cpu')['model_state']
C:\U


  M2_E1: after_A -> after_A_then_B
  Layer                                    Norm delta  Rel delta
  ---------------------------------------- ---------- ----------
  token_emb.weight                            69.0486     0.4997
  pos_emb.weight                               4.7511     0.6015
  blocks.0.norm1.weight                        1.5232     0.1069
  blocks.0.norm1.bias                          0.2576     0.9163
  blocks.0.attn.qkv.weight                     5.8577     0.3335
  blocks.0.attn.out_proj.weight                2.2509     0.1533
  blocks.0.norm2.weight                        0.7887     0.0518
  blocks.0.norm2.bias                          0.0468     0.8868
  blocks.0.ff.0.weight                         5.5838     0.2798
  blocks.0.ff.0.bias                           0.2144     0.1886
  blocks.0.ff.3.weight                         7.4220     0.3925
  blocks.0.ff.3.bias                           0.0805     0.2753
  blocks.1.norm1.weight                        1.2061 

...


  M2_E3: after_B -> after_B_then_A
  Layer                                    Norm delta  Rel delta
  ---------------------------------------- ---------- ----------
  token_emb.weight                            99.6658     1.3026
  pos_emb.weight                               3.3388     0.4612
  blocks.0.norm1.weight                        1.3571     0.0957
  blocks.0.norm1.bias                          0.1612     0.4946
  blocks.0.attn.qkv.weight                     5.3790     0.3087
  blocks.0.attn.out_proj.weight                2.0520     0.1408
  blocks.0.norm2.weight                        0.6896     0.0453
  blocks.0.norm2.bias                          0.0546     0.7992
  blocks.0.ff.0.weight                         4.9244     0.2470
  blocks.0.ff.0.bias                           0.2123     0.1875
  blocks.0.ff.3.weight                         6.7827     0.3558
  blocks.0.ff.3.bias                           0.0794     0.2579
  blocks.1.norm1.weight                        1.0863 

C:\Users\rajcr\AppData\Local\Temp\ipykernel_31388\2332656341.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s2 = torch.load(p2, map_location='cpu')['model_state']



  M3_E1: after_A -> after_A_then_B
  Layer                                    Norm delta  Rel delta
  ---------------------------------------- ---------- ----------
  token_emb.weight                            56.5072     0.4876
  pos_emb.weight                               3.6069     0.3459
  blocks.0.norm1.weight                        0.8929     0.0479
  blocks.0.norm1.bias                          0.1957     1.1129
  blocks.0.attn.qkv.weight                     5.3143     0.2336
  blocks.0.attn.out_proj.weight                2.0698     0.1105
  blocks.0.norm2.weight                        0.4472     0.0235
  blocks.0.norm2.bias                          0.0452     0.8811
  blocks.0.ff.0.weight                         4.5565     0.1871
  blocks.0.ff.0.bias                           0.1017     0.0892
  blocks.0.ff.3.weight                         5.4256     0.2274
  blocks.0.ff.3.bias                           0.0638     0.2265
  blocks.1.norm1.weight                        0.7922 

C:\Users\rajcr\AppData\Local\Temp\ipykernel_31388\2332656341.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  s1 = torch.load(p1, map_location='cpu')['model_state']
C:\U


  M3_E3: after_B -> after_B_then_A
  Layer                                    Norm delta  Rel delta
  ---------------------------------------- ---------- ----------
  token_emb.weight                            90.6305     1.3865
  pos_emb.weight                               2.6757     0.2700
  blocks.0.norm1.weight                        0.9388     0.0504
  blocks.0.norm1.bias                          0.1149     0.5921
  blocks.0.attn.qkv.weight                     4.7850     0.2110
  blocks.0.attn.out_proj.weight                1.8977     0.1014
  blocks.0.norm2.weight                        0.4678     0.0246
  blocks.0.norm2.bias                          0.0375     0.6692
  blocks.0.ff.0.weight                         4.0514     0.1667
  blocks.0.ff.0.bias                           0.0957     0.0862
  blocks.0.ff.3.weight                         4.7646     0.1991
  blocks.0.ff.3.bias                           0.0579     0.1979
  blocks.1.norm1.weight                        0.7492 